# ==========================================
# HOUSING PRICES - ADVANCED STACKING
# ==========================================
# Strategy: Combine 4 diverse models to maximize performance
#   - XGBoost (shallow): Best single model so far
#   - LightGBM: Fast gradient boosting
#   - CatBoost: Handles categorical features natively
#   - Random Forest: Different approach (bagging)
# Meta-model: Ridge Regression
# ==========================================


In [1]:
# ==========================================
# 1. IMPORTS
# ==========================================
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
# ==========================================
# 2. LOAD DATA
# ==========================================
train_data = pd.read_csv('/kaggle/input/competitions/home-data-for-ml-course/train.csv')
test_data = pd.read_csv('/kaggle/input/competitions/home-data-for-ml-course/test.csv')

test_ids = test_data['Id'].copy()
train_data.set_index('Id', inplace=True)
test_data.set_index('Id', inplace=True)

print(f"Train shape: {train_data.shape}")
print(f"Test shape: {test_data.shape}")


Train shape: (1460, 80)
Test shape: (1459, 79)


In [3]:
# ==========================================
# 3. DROP COLUMNS WITH >1000 MISSING VALUES
# ==========================================
cols_to_drop = train_data.columns[train_data.isnull().sum() > 1000].tolist()
train_data.drop(cols_to_drop, axis=1, inplace=True)
test_data.drop(cols_to_drop, axis=1, inplace=True)
print(f"Dropped {len(cols_to_drop)} columns")


Dropped 4 columns


In [4]:
# ==========================================
# 4. FEATURE ENGINEERING
# ==========================================
for df in [train_data, test_data]:
    df['TotalSF'] = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF']
    df['TotalBath'] = df['FullBath'] + 0.5*df['HalfBath'] + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath']
    df['Age'] = df['YrSold'] - df['YearBuilt']
    df['IsRemodeled'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)
    df['TotalPorch'] = df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch']

print(f"After Feature Engineering: {train_data.shape[1]} columns")


After Feature Engineering: 81 columns


In [5]:
# ==========================================
# 5. HANDLE MISSING VALUES
# ==========================================
nums_train = train_data.select_dtypes(include=['int64', 'float64']).columns
cats_train = train_data.select_dtypes(include=['object']).columns

for col in nums_train:
    train_data[col] = train_data[col].fillna(train_data[col].median())
for col in cats_train:
    train_data[col] = train_data[col].fillna(train_data[col].mode()[0])

for col in nums_train:
    if col in test_data.columns:
        test_data[col] = test_data[col].fillna(train_data[col].median())
for col in cats_train:
    if col in test_data.columns:
        test_data[col] = test_data[col].fillna(train_data[col].mode()[0])

print(f"Missing in train: {train_data.isnull().sum().sum()}")
print(f"Missing in test: {test_data.isnull().sum().sum()}")


Missing in train: 0
Missing in test: 0


In [6]:
# ==========================================
# 6. ENCODE CATEGORICAL VARIABLES
# ==========================================
X = train_data.drop('SalePrice', axis=1)
y = train_data['SalePrice']

X_encoded = pd.get_dummies(X, drop_first=True)
test_encoded = pd.get_dummies(test_data, drop_first=True)
test_encoded = test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

print(f"Final feature count: {X_encoded.shape[1]}")


Final feature count: 240


In [7]:
# ==========================================
# 7. DEFINE 3 DIVERSE MODELS (KNN version)
# ==========================================
models = {
    'CatBoost_Deep': CatBoostRegressor(
        iterations=1500,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3.0,
        random_state=42,
        verbose=0
    ),
    'XGB_Shallow': XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
        n_jobs=-1
    ),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),  # KNN needs scaling!
        ('knn', KNeighborsRegressor(
            n_neighbors=10,
            weights='distance',        # Closer neighbors matter more
            n_jobs=-1
        ))
    ])
}

print(f"\nDefined {len(models)} diverse models:")
for name in models:
    print(f"  - {name}")


Defined 3 diverse models:
  - CatBoost_Deep
  - XGB_Shallow
  - KNN


In [8]:
# ==========================================
# 8. STACKING WITH OUT-OF-FOLD PREDICTIONS
# ==========================================
print("\n" + "="*50)
print("🔧 Training Advanced Stacking...")
print("="*50)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_predictions = {name: np.zeros(len(X_encoded)) for name in models}
test_predictions = {name: np.zeros(len(test_encoded)) for name in models}

for fold, (train_idx, val_idx) in enumerate(kf.split(X_encoded)):
    print(f"\n📁 Fold {fold + 1}/5")
    
    X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        oof_predictions[name][val_idx] = model.predict(X_val)
        test_predictions[name] += model.predict(test_encoded) / kf.n_splits
        
        fold_mae = mean_absolute_error(y_val, oof_predictions[name][val_idx])
        print(f"   {name}: MAE = {fold_mae:,.2f}")



🔧 Training Advanced Stacking...

📁 Fold 1/5
   CatBoost_Deep: MAE = 15,972.69
   XGB_Shallow: MAE = 15,357.48
   KNN: MAE = 24,345.70

📁 Fold 2/5
   CatBoost_Deep: MAE = 13,103.47
   XGB_Shallow: MAE = 13,878.32
   KNN: MAE = 27,790.98

📁 Fold 3/5
   CatBoost_Deep: MAE = 16,153.68
   XGB_Shallow: MAE = 17,947.62
   KNN: MAE = 25,579.43

📁 Fold 4/5
   CatBoost_Deep: MAE = 15,270.15
   XGB_Shallow: MAE = 16,461.08
   KNN: MAE = 25,647.57

📁 Fold 5/5
   CatBoost_Deep: MAE = 12,844.73
   XGB_Shallow: MAE = 13,620.59
   KNN: MAE = 21,975.70


In [9]:
# ==========================================
# 9. EVALUATE EACH BASE MODEL
# ==========================================
print("\n" + "="*50)
print("📊 Base Models (Out-of-Fold MAE):")
print("="*50)
for name in models:
    mae = mean_absolute_error(y, oof_predictions[name])
    print(f"{name}: {mae:,.2f}")



📊 Base Models (Out-of-Fold MAE):
CatBoost_Deep: 14,668.95
XGB_Shallow: 15,453.02
KNN: 25,067.87


In [10]:
# ==========================================
# 10. META-MODEL (STACKING)
# ==========================================
stacked_train = np.column_stack([oof_predictions[name] for name in models])
stacked_test = np.column_stack([test_predictions[name] for name in models])

meta_model = Ridge(alpha=1.0)
meta_model.fit(stacked_train, y)

stacked_pred = meta_model.predict(stacked_train)
stacked_mae = mean_absolute_error(y, stacked_pred)

print("\n" + "="*50)
print("🏆 STACKING RESULT:")
print("="*50)
print(f"Stacked MAE: {stacked_mae:,.2f}")

print("\nMeta-model weights:")
for name, weight in zip(models.keys(), meta_model.coef_):
    print(f"  {name}: {weight:.4f}")



🏆 STACKING RESULT:
Stacked MAE: 14,552.79

Meta-model weights:
  CatBoost_Deep: 0.8748
  XGB_Shallow: 0.1278
  KNN: 0.0288


In [11]:
# ==========================================
# 11. FINAL PREDICTIONS ON TEST
# ==========================================
final_predictions = meta_model.predict(stacked_test)


In [12]:
# ==========================================
# 12. CREATE SUBMISSION FILE
# ==========================================
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})

print(f"\n✅ Submission rows: {len(submission)} (expected: 1459)")
submission.to_csv('submission.csv', index=False)
print("\n✅ submission.csv created successfully!")


✅ Submission rows: 1459 (expected: 1459)

✅ submission.csv created successfully!
